# logit-thermo A2-7c：本滴定のやり直し（Qwen2.5-1.5B、GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは約 1.5〜2 時間。終わったら自動で切れます。
- ゲート：`docs/A2_7C_GATE.md`。A2-7b の Qwen だけを、温度 0.6〜1.05（A2-7 と同じ 8 つ）でやり直す。腕は A2-7b と同じ 8 本。
- 計算のコード：A2-2・A2-4・A2-7・A2-7b のデータ集めのコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_7c_collect.py`（`0bb013deef86458f…`）は埋め込み。
- 保存先 `A2_7C/`。途中で切れても、もう一度「すべて実行」すれば終わった腕は飛ばします。
- **通知**：🚀・腕ごとの ✅・🎉・⚠️。名札は「logit-thermo A2-7c 本滴定やり直し データ集め（Qwen2.5-1.5B、温度 0.6〜1.05、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-7c 本滴定やり直し データ集め（Qwen2.5-1.5B、温度 0.6〜1.05、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-7c titration qwen redo', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0'})
SRC22, SRC24, SRC7C, SRC7BC = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC']
SRC7CC = '"""A2-7c（ゲート docs/A2_7C_GATE.md）：A2-7b の Qwen だけのやり直し。温度の格子を A2-7 と同じ 0.6〜1.05 に戻す。GPU（L4）。\nscripts/a2_7b_collect.py を実行したあとの同じ名前空間で実行し、温度の並びだけを置きかえる（関数は実行するときに B7_TEMPS を読むので、\n腕・種の決め方・禁止のしかたは A2-7b のまま）。生成の種の k は、この新しい 8 つの温度での番号。\n"""\nB7_TEMPS = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05]\nC7_GROUP = \'qwen\'\n\n\ndef self_test_7c():\n    assert B7_TEMPS == [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05]\n    assert all(T in B7_TEMPS for T in [0.8, 0.85, 0.9])\n    assert [a[0] for a in B7_ARMS] == [\'none\', \'none_b\', \'cq33\', \'cq67\', \'cq100\', \'pl025\', \'pl050\', \'pl075\']\n    assert B7_GROUPS[C7_GROUP] == (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\')\n    return \'OK\'\n'
assert hashlib.sha256(SRC7CC.encode()).hexdigest() == '0bb013deef86458fb73c2acf9b41e21d41328ff22c3ea64c467686eeaf4c9751'
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRC7CC)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 8 本の腕 → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_7c())
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_7C', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    name, hf_id = B7_GROUPS[C7_GROUP]
    run_7b(name, hf_id, docs, '/content/drive/MyDrive/logit-thermo/A2_7C', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(gate='docs/A2_7C_GATE.md', group=C7_GROUP, temps=B7_TEMPS, a2_7b_collect_sha256='b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0',
                   a2_7c_collect_sha256='0bb013deef86458fb73c2acf9b41e21d41328ff22c3ea64c467686eeaf4c9751', torch=torch.__version__, transformers=transformers.__version__,
                   python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_7C' + '/run_meta_qwen.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-7c のデータ集め（Qwen2.5-1.5B）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：A2-7c の Qwen2.5-1.5B を A2_7C に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
